# 2. compare suburbs

In [0]:
%sql
CREATE OR REPLACE TABLE cpt_utility_catalog.gold.dim_suburb AS

WITH distinct_arrears_suburbs AS (
    -- Get distinct suburb names from suburb arrears
    SELECT DISTINCT 
        UPPER(TRIM(suburb)) AS suburb_name
    FROM cpt_utility_catalog.silver.silver_suburb_arrears_cleaned
    WHERE suburb IS NOT NULL
),

dominant_wards AS (
    -- Match suburbs with service requests to get the primary ward (mode)
    SELECT 
        UPPER(TRIM(suburb)) AS suburb_name,
        CAST(ward AS INT) AS ward,
        ROW_NUMBER() OVER (
            PARTITION BY UPPER(TRIM(suburb)) 
            ORDER BY COUNT(*) DESC
        ) AS rank
    FROM cpt_utility_catalog.silver.silver_service_requests_cleaned
    WHERE suburb IS NOT NULL AND ward IS NOT NULL
    GROUP BY UPPER(TRIM(suburb)), CAST(ward AS INT)
),

mapped_subcouncils AS (
    -- Map ward numbers to subcouncils
    SELECT 
        a.suburb_name,
        COALESCE(w.ward_number, -1) AS ward_number,
        CASE 
            -- Subcouncil 1
            WHEN w.ward_number IN (23, 29, 32, 107) THEN 1
            -- Subcouncil 2
            WHEN w.ward_number IN (6, 7, 8, 101, 102, 111) THEN 'SUBCOUNCIL 02'
            -- Subcouncil 3
            WHEN w.ward_number IN (1, 4, 5, 70, 105, 113) THEN 'SUBCOUNCIL 03'
            -- Subcouncil 4
            WHEN w.ward_number IN (25, 26, 27, 28) THEN 'SUBCOUNCIL 04'
            -- Subcouncil 5
            WHEN w.ward_number IN (13, 20, 24, 31, 50, 106) THEN 'SUBCOUNCIL 05'
            -- Subcouncil 6
            WHEN w.ward_number IN (2, 3, 9, 10) THEN 'SUBCOUNCIL 06'
            -- Subcouncil 7
            WHEN w.ward_number IN (21, 103, 105, 112) THEN 'SUBCOUNCIL 07'
            -- Subcouncil 8
            WHEN w.ward_number IN (15, 83, 84, 85, 86, 100) THEN 'SUBCOUNCIL 08'
            -- Subcouncil 9
            WHEN w.ward_number IN (87, 89, 90, 91, 116) THEN 'SUBCOUNCIL 09'
            -- Subcouncil 10
            WHEN w.ward_number IN (92, 93, 94, 97, 98, 99) THEN 'SUBCOUNCIL 10'
            -- Subcouncil 11
            WHEN w.ward_number IN (40, 44, 46, 47) THEN 'SUBCOUNCIL 11'
            -- Subcouncil 12
            WHEN w.ward_number IN (78, 79, 81, 82) THEN 'SUBCOUNCIL 12'
            -- Subcouncil 13
            WHEN w.ward_number IN (34, 35, 36, 80, 88) THEN 'SUBCOUNCIL 13'
            -- Subcouncil 14
            WHEN w.ward_number IN (30, 37, 38, 39, 41, 42, 45) THEN 'SUBCOUNCIL 14'
            -- Subcouncil 15
            WHEN w.ward_number IN (51, 52, 53, 56) THEN 'SUBCOUNCIL 15'
            -- Subcouncil 16
            WHEN w.ward_number IN (54, 57, 77, 115) THEN 'SUBCOUNCIL 16'
            -- Subcouncil 17
            WHEN w.ward_number IN (48, 49, 60) THEN 'SUBCOUNCIL 17'
            -- Subcouncil 18
            WHEN w.ward_number IN (65, 66, 67, 68, 110) THEN 'SUBCOUNCIL 18'
            -- Subcouncil 19
            WHEN w.ward_number IN (61, 64, 69) THEN 'SUBCOUNCIL 19'
            -- Subcouncil 20
            WHEN w.ward_number IN (58, 59, 62, 63, 71, 73, 74) THEN 'SUBCOUNCIL 20'
            -- Subcouncil 21
            WHEN w.ward_number IN (11, 19, 108) THEN 'SUBCOUNCIL 21'
            -- Subcouncil 22
            WHEN w.ward_number IN (14, 16, 17, 114) THEN 'SUBCOUNCIL 22'
            -- Subcouncil 23
            WHEN w.ward_number IN (33, 75, 76) THEN 'SUBCOUNCIL 23'
            -- Subcouncil 24
            WHEN w.ward_number IN (12, 18, 95, 96) THEN 'SUBCOUNCIL 24'
            ELSE 'UNKNOWN'
        END AS subcouncil
    FROM distinct_arrears_suburbs a
    LEFT JOIN dominant_wards w 
           ON a.suburb_name = w.suburb_name 
          AND w.rank = 1
)

-- STEP 4: Map subcouncil to city region
SELECT 
    xxhash64(LOWER(suburb_name)) AS suburb_key,
    suburb_name,
    CASE WHEN ward_number = -1 THEN 'UNKNOWN' ELSE CAST(ward_number AS STRING) END AS ward,
    subcouncil,
    CASE 
        WHEN subcouncil IN ('SUBCOUNCIL 01', 'SUBCOUNCIL 03') THEN 'BLOUBERG / WEST COAST'
        WHEN subcouncil IN ('SUBCOUNCIL 02', 'SUBCOUNCIL 04', 'SUBCOUNCIL 06', 'SUBCOUNCIL 07') THEN 'NORTHERN SUBURBS'
        WHEN subcouncil IN ('SUBCOUNCIL 05', 'SUBCOUNCIL 09', 'SUBCOUNCIL 10', 'SUBCOUNCIL 11', 'SUBCOUNCIL 12', 'SUBCOUNCIL 13', 'SUBCOUNCIL 14', 'SUBCOUNCIL 17', 'SUBCOUNCIL 23') THEN 'CAPE FLATS'
        WHEN subcouncil IN ('SUBCOUNCIL 08', 'SUBCOUNCIL 21', 'SUBCOUNCIL 22', 'SUBCOUNCIL 24') THEN 'HELDERBERG & EASTERN'
        WHEN subcouncil IN ('SUBCOUNCIL 15', 'SUBCOUNCIL 18', 'SUBCOUNCIL 20') THEN 'SOUTHERN SUBURBS'
        WHEN subcouncil = 'SUBCOUNCIL 16' THEN 'CITY BOWL & ATLANTIC SEABOARD'
        WHEN subcouncil = 'SUBCOUNCIL 19' THEN 'SOUTHERN PENINSULA'
        ELSE 'UNASSIGNED'
    END AS city_region
FROM mapped_subcouncils;